# IM2 hillslope routing over a day: multi-step property tests

`hillslope_im2.ipynb` showed, one call of `HydrologyDrainage` at a time, that ELM's
`set_topounit` gives the lowest topounit of some gridcells a downhill neighbour (itself, or an
uphill one), and that in a wet step none of the surface runoff of such a hillslope reaches the
river. A single call cannot show what happens next: water a topounit sends waits in the downhill
topounit's store (`top_ws%from_uphill`), half of it is released into that topounit's soil each step
(`SurfaceRunoff`, in `HydrologyNoDrainage`), and the store is drained in `ColWaterBalanceCheck`.

This notebook runs both hydrology routines for a day (48 half-hour steps), with SPEL proptest's
multi-step runs (`spel/proptest/SPEC.md`, "Multi-step runs"), and asks:

1. Which variables does the rest of ELM's step change between two calls (the *boundary*), and what
   stands in for it here?
2. Do the store's invariants hold over a day: finite, conserved, released at the documented rate?
3. Compared with the same example under IM2 off: what must be the same, and what must differ in a
   known direction? Which of these does the `downhill_ti` bug break?
4. How much of a hillslope's runoff reaches the river over a storm, with and without the bug?

The spec is built by `hillslope_im2_season_spec.py` (this directory) and written as
`hillslope_im2_season.proptest.toml`, so `spel proptest run` runs the same checks.

## 0. Setup

The case `hs_season` has the two routines, `hydrologynodrainage` and `hydrologydrainage`, in
elm_drv's order (`spel create -s hydrologynodrainage hydrologydrainage -c hs_season`, from the `next`
worktree of E3SM master). Its reference, `unit-tests/input-data/hs_season`, is the union of the
`hs_bug_nodrain` and `hs_bug` captures of the same ELM run (`hs-im2`, the 7 gridcells with a
self-looping outlet): their 19 shared constants are identical, and for shared inputs the
no-drainage capture (the state at the start of the step) is used. It supplies constants,
parameters, level extents and the statistics behind the input rules; it is not unit-test reference
data (`spel_reference.json` says so).

In [1]:
import os
os.environ.setdefault("SPEL_E3SM_SRCROOT", "/lcrc/globalscratch/ac.schwartzpd/next")

import copy
import io
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from spel.proptest import coupling, runner
from spel.proptest import spec as sp

import hillslope_im2_season_spec as season

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", None)
HERE = Path.cwd()

## 1. The boundary: what the rest of the step does

Between two calls ELM runs the rest of its step. `coupling.boundary` finds, in the static trace of
elm_drv over the full analysis (the same trace the database's call-tree view uses), every write to
a variable the case's routines read that happens outside them, under these namelist options and on
filters that can share an element with the case's own accesses:

In [2]:
spec = season.build(HERE / "hillslope_im2_season.proptest.toml")
spec.to_toml(spec.path)
case = spec.case
IM2_VALUES = {"use_im2_hillslope_hydrology": True}
live = case.inv.configured(IM2_VALUES)
found = coupling.boundary(case.inv, IM2_VALUES)
boundary = {n: w for n, w in found.items() if live.vars[n].kind in ("input", "inout")}
print(f"{len(boundary)} variables the routines read are written elsewhere in the step")
rows = [dict(variable=n, kind=live.vars[n].kind, writers=len(w), where=coupling.describe(w, 1))
        for n, w in sorted(boundary.items())]
pd.DataFrame(rows).set_index("variable").loc[["top_ws%from_uphill", "col_wf%qflx_rain_grnd", "col_ws%h2osoi_liq",
                                             "col_es%t_soisno", "col_wf%qflx_evap_tot", "col_pp%dz"]]

61 variables the routines read are written elsewhere in the step


,kind,writers,where
variable,,,
top_ws%from_uphill,inout,2,"colwaterbalancecheck (BalanceCheckMod.F90:322), after; and 1 more"
col_wf%qflx_rain_grnd,input,1,"p2c_1d_filter (subgridAveMod.F90:333) in elm_drv_patch2col, before"
col_ws%h2osoi_liq,inout,6,"copy_state (dynInitColumnsMod.F90:284) in dynsubgrid_driver, before, conditional; and 5 more"
col_es%t_soisno,inout,21,"copy_state (dynInitColumnsMod.F90:276) in dynsubgrid_driver, before, conditional; and 20 more"
col_wf%qflx_evap_tot,input,1,"p2c_1d_filter (subgridAveMod.F90:333) in elm_drv_patch2col, before"
col_pp%dz,inout,3,"canopyhydrology (CanopyHydrologyMod.F90:670), before, conditional; and 2 more"


`qflx_surf` and `qflx_from_uphill` are not on the list: with both routines in the case, only the
case writes them, so they are carried from step to step. `top_ws%from_uphill` is written after
the case, by `ColWaterBalanceCheck` (the drain of the store). A multi-step scenario must say what
stands in for each boundary variable, and the run refuses to start otherwise:

* `forcing`: set before every step. Here the rain on the ground (`qflx_rain_grnd`): in the storm
  scenario a drawn intensity for the first 12 steps (6 hours), then 0.
* `between`: assigned after every step. Here the store's drain, ColWaterBalanceCheck's three lines
  written as an expression: `from_uphill(t) - dtime_mod * child_sum(qflx_from_uphill, 't')`, at
  least 0, and 0 below 1e-20.
* `keep`: left to the process. Everything else: soil water and temperature, snow, canopy and the
  other fluxes. The inputs here make that a statement about an isolated process: no snow, ice,
  evaporation or transpiration (those fluxes are 0), soil warmer than freezing, no flooding.

In [3]:
parsed = spec.validate()
summary = []
for name, sc in parsed.scenarios.items():
    declared = Counter("forcing" if (n in sc.forcing or any(v == n for v, _ in sc.forcing_exprs))
                       else "between" if any(v == n for v, _ in sc.between)
                       else "keep" if sc.boundary_declared(n) else "UNDECLARED" for n in boundary)
    summary.append(dict(scenario=name, steps=sc.steps, **declared))
print("between:", parsed.scenarios["storm"].between[0][1].text)
print("storm forcing:", parsed.scenarios["storm"].forcing_exprs[0][1].text)
pd.DataFrame(summary).set_index("scenario").fillna(0).astype(int)

between: merge(0.0, max(0.0, top_ws%from_uphill(t) - dtime_mod * child_sum(col_wf%qflx_from_uphill, 't')), max(0.0, top_ws%from_uphill(t) - dtime_mod * child_sum(col_wf%qflx_from_uphill, 't')) < 1.0e-20)
storm forcing: merge(col_wf%qflx_rain_grnd(c), 0.0, step <= 12)


,steps,keep,forcing,between
scenario,,,,
storm,48,59,1,1
steady,48,60,0,1
recession,48,60,0,1


## 2. The hillslopes

Four gridcells from the first notebook (soil topounits with declared elevations, two sets): the
right order, its mirror image, the 4-topounit loop and the real Caspian gridcell; and a gridcell of
one topounit (`flat`), where IM2 must change nothing. Besides ELM's `downhill_ti`, the spec uses
three functions of the domain declaration: `elevation(t)`, `lowest(t)` and `uphill_count(t)` (how
many other topounits point at t):

In [4]:
s = spec.session()
lay = s.layout
tt = lay.table("t")
t_idx = np.arange(1, lay.count("t") + 1)
fns = s.domain_functions()
tt["elevation"] = fns["elevation"](t_idx)
tt["downhill_ti"] = np.asarray(s.static("top_pp%downhill_ti").data)
tt["lowest"] = fns["lowest"](t_idx)
tt["uphill_count"] = fns["uphill_count"](t_idx)
print("built domain vs declaration:", s.built().check() or "no differences")
tt[tt["set"] == 0][["name", "topo_number", "weight", "elevation", "downhill_ti", "lowest", "uphill_count"]]

built domain vs declaration: no differences


,name,topo_number,weight,elevation,downhill_ti,lowest,uphill_count
topounit,,,,,,,
1,lowest first,1,0.30,50.0,-1,True,1
2,lowest first,2,0.30,150.0,1,False,1
3,lowest first,3,0.40,300.0,2,False,0
4,lowest last,1,0.40,300.0,5,False,0
5,lowest last,2,0.30,150.0,6,False,1
6,lowest last,3,0.30,50.0,6,True,1
7,cycle,1,0.25,300.0,8,False,0
8,cycle,2,0.25,200.0,10,False,1
9,cycle,3,0.25,50.0,10,True,1


## 3. One storm

One drawn example of the storm scenario, run with IM2 on and, in a sibling session, off. The store of
each topounit of "lowest first" and of its mirror "lowest last" every 4 steps (kg/m² of topounit):

In [5]:
ex = s.sample("storm", seed=1)
on = s.execute("storm", ex)
off_session = s.sibling({"use_im2_hillslope_hydrology": False})
off = off_session.execute("storm", ex)
store = on.series("top_ws%from_uphill")
g_t = lay.ancestor_of("t", "g")
names_t = tt["name"].to_numpy()
cols = {f"{n} t{k}": i for i, (n, k, st) in enumerate(zip(names_t, tt["topo_number"], tt["set"]))
        if st == 0 and n in ("lowest first", "lowest last")}
frame = pd.DataFrame({c: store[:, i] for c, i in cols.items()}, index=pd.RangeIndex(1, store.shape[0] + 1, name="step"))
frame.iloc[::4]

,lowest first t1,lowest first t2,lowest first t3,lowest last t1,lowest last t2,lowest last t3
step,,,,,,
1,1.6310e-01,2.2203e-02,0.0,0.0,2.1828e-01,1.2859e-01
5,1.9665e-01,4.3018e-02,0.0,0.0,2.3231e-01,2.4915e-01
9,2.1334e-01,4.4319e-02,0.0,0.0,2.3261e-01,2.5669e-01
13,1.1088e-01,2.2197e-02,0.0,0.0,1.1610e-01,1.2856e-01
17,6.9301e-03,1.3873e-03,0.0,0.0,7.2564e-03,8.0352e-03
21,4.3313e-04,8.6709e-05,0.0,0.0,4.5353e-04,5.0220e-04
25,2.7071e-05,5.4193e-06,0.0,0.0,2.8345e-05,3.1388e-05
29,1.6919e-06,3.3871e-07,0.0,0.0,1.7716e-06,1.9617e-06
33,1.0574e-07,2.1169e-08,0.0,0.0,1.1072e-07,1.2261e-07


The routing is visible in the stores. In "lowest first" the two upper topounits send their surface
runoff down: t1 (the outlet, 50 m) collects from t2, and t2 from t3; t3 (the top) receives nothing.
After the rain stops (step 12) the stores empty within a few hours, half of what is left each step.
In "lowest last" the outlet is t3 and it points at itself: t1 (the top) receives nothing, t2 collects
from t1, and the outlet's store fills with its own runoff as well as t2's.

Per gridcell, the runoff that reached the river so far (kg/m² of gridcell), IM2 on and off:

In [6]:
DT = 1800.0
w_c, g_c = lay.weight("c", "g"), lay.ancestor_of("c", "g")
gt = lay.table("g")


def river(res):
    # cumulative runoff of each gridcell (kg/m2 of gridcell) after each step
    r = res.series("col_wf%qflx_runoff") * w_c * DT
    per_g = np.stack([np.bincount(g_c, row, lay.count("g")) for row in r])
    return np.cumsum(per_g, axis=0)


r_on, r_off = river(on), river(off)
last = pd.DataFrame({"hillslope": gt["name"], "set": gt["set"], "IM2 on": r_on[-1], "IM2 off": r_off[-1]})
last["on / off"] = last["IM2 on"] / last["IM2 off"]
last[last["set"] == 0].set_index("hillslope")

,set,IM2 on,IM2 off,on / off
hillslope,,,,
lowest first,0,75.6512,76.0859,0.9943
lowest last,0,65.7338,66.6464,0.9863
cycle,0,88.2999,92.1463,0.9583
caspian 184059,0,5.3199,5.8209,0.9139
flat,0,0.1920,0.1920,1.0000


## 4. The checks

Properties of the store, and relations between the same example with IM2 on (`base%`) and off
(`follow%`), judged after every one of the 48 steps (SPEC.md: `tend(x)` = the step's change,
`acc(x)` = the sum over steps so far, `child_wsum(x, 'g')` = the gridcell's weighted sum):

In [7]:
pd.DataFrame([dict(check=c.name, kind=c.table, scenarios=", ".join(c.scenarios), expression=c.judged.text)
              for c in parsed.checks]).set_index("check")

,kind,scenarios,expression
check,,,
the store stays finite and non-negative,property,*,finite(top_ws%from_uphill(t)) .and. top_ws%from_uphill(t) >= 0.0
no topounit sends water to itself,property,*,"implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)"
the stores gain what is sent and lose what is released,property,*,"close(tend(child_wsum(top_ws%from_uphill, 'g')), dtime_mod * (child_wsum(col_wf%qflx_to_downhill, 'g') - child_wsum(col_wf%qflx_from_uphill, 'g')), rtol=1.0e-9, atol=1.0e-12)"
a topounit releases half its store per step,property,*,"close(dtime_mod * child_sum(col_wf%qflx_from_uphill, 't'), 0.5 * prev%top_ws%from_uphill(t), rtol=1.0e-12, atol=1.0e-15)"
the topmost topounit's store halves every step in a recession,property,recession,"implies(uphill_count(t) == 0, close(top_ws%from_uphill(t), 0.5 * prev%top_ws%from_uphill(t), rtol=1.0e-12, atol=1.0e-20))"
R1 a gridcell of one topounit is the same with IM2 off,relation,"storm, steady","implies(top_pp%downhill_ti(t) == -1 .and. uphill_count(t) == 0, all(same(follow%col_ws%h2osoi_liq(c, 1:nlevgrnd), base%col_ws%h2osoi_liq(c, 1:nlevgrnd))) .and. same(follow%col_ws%h2osfc(c), base%col_ws%h2osfc(c)) .and. same(follow%col_wf%qflx_runoff(c), base%col_wf%qflx_runoff(c)))"
R2 the soil of the topmost topounit is the same with IM2 off,relation,"storm, steady","implies(uphill_count(t) == 0, all(same(follow%col_ws%h2osoi_liq(c, 1:nlevgrnd), base%col_ws%h2osoi_liq(c, 1:nlevgrnd))) .and. same(follow%col_ws%h2osfc(c), base%col_ws%h2osfc(c)))"
R4 water is conserved: soil + stores + runoff so far are the same with IM2 off,relation,"storm, steady","close(child_wsum(base%col_ws%endwb, 'g') + child_wsum(base%top_ws%from_uphill, 'g') + dtime_mod * acc(child_wsum(base%col_wf%qflx_runoff, 'g')), child_wsum(follow%col_ws%endwb, 'g') + dtime_mod * acc(child_wsum(follow%col_wf%qflx_runoff, 'g')), rtol=1.0e-9, atol=1.0e-9)"
D1 IM2 never gets water to the river sooner,relation,"storm, steady","acc(child_wsum(base%col_wf%qflx_runoff, 'g')) <= acc(child_wsum(follow%col_wf%qflx_runoff, 'g')) * (1.0 + 1.0e-12) + 1.0e-15"


`runner.run` gives every check a Hypothesis test (6 examples each here, seed 0); the IM2-off runs are a
sibling session, stepping in parallel with the base run.

In [8]:
t0 = time.time()
outcomes = runner.run(parsed, case.inv, settings={"seed": 0, "max_examples": 6}, session=s, out=io.StringIO())
print(f"{time.time() - t0:.0f} s")
results = pd.DataFrame([(o.scenario, o.check, "pass" if o.passed else "FAIL", round(o.seconds))
                        for o in outcomes], columns=["scenario", "check", "result", "seconds"])
results.pivot(index="check", columns="scenario", values="result").fillna("")

351 s


scenario,recession,steady,storm
check,,,
D1 IM2 never gets water to the river sooner,,pass,pass
D2 a topounit that receives water holds at least as much,,pass,pass
D3 the outlet passes on at least what it does without IM2,,FAIL,FAIL
R1 a gridcell of one topounit is the same with IM2 off,,pass,pass
R2 the soil of the topmost topounit is the same with IM2 off,,pass,pass
R4 water is conserved: soil + stores + runoff so far are the same with IM2 off,,pass,pass
a topounit releases half its store per step,pass,pass,pass
no topounit sends water to itself,pass,FAIL,FAIL
the store stays finite and non-negative,pass,pass,pass


In [9]:
for o in outcomes:
    if not o.passed:
        print("\n".join(o.message.splitlines()[:7]), "\n")

no topounit sends water to itself: implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)
  after step 1 of 48
  fails at 4 of 30 elements; first: element 6 (set 0, lowest last, topounit 3/3, soil, column soil)
    implies = False
    col_wf%qflx_to_downhill = 4.306334178454746e-05
    top_pp%downhill_ti = 6
  drawn inputs at that element: 

D3 the outlet passes on at least what it does without IM2: implies(lowest(t), acc(base%col_wf%qflx_runoff(c)) >= acc(follow%col_wf%qflx_runoff(c)) * (1.0 - 1.0e-12) - 1.0e-15)
  after step 1 of 48
  fails at 6 of 30 elements; first: element 6 (set 0, lowest last, topounit 3/3, soil, column soil)
    implies = False
    lowest = True
    acc = -5.000000000000837e-05
    base%col_wf%qflx_runoff = -5.000000000000837e-05 

no topounit sends water to itself: implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)
  after step 1 of 48
  fails at 4 of 30 elements; first: element 6 (set 0, lowest last, topounit 3/3, soil, colum

Everything that must hold whatever the topology holds, in every scenario: the store stays finite and
non-negative, gains exactly what is sent and loses what is released, releases half of itself per step,
and in a recession the topmost store halves exactly. With IM2 off, a one-topounit gridcell and the
topmost topounit's soil are bit-for-bit the same, total water (soil, stores and runoff so far) is
conserved, IM2 never gets water to the river sooner, and receiving topounits hold at least as much.

Two checks fail, in storm and steady, and only where the bug acts: no topounit may send water to
itself, and the outlet must pass on at least what it does without IM2 (D3). The failures print the
first failing element; the verdicts list them all. On one storm example:

In [10]:
d3 = next(c for c in parsed.checks if c.name.startswith("D3"))
selfsend = next(c for c in parsed.checks if c.name.startswith("no topounit"))
storm = parsed.scenarios["storm"]
t_of_c = lay.ancestor_of("c", "t")
rows = []
for check in (selfsend, d3):
    v = s.evaluate(storm, check, check.prepare(s, storm, ex))  # the first step at which it fails
    rows.append(dict(check=check.name, first_failing_step=v.step, failing=len(v.failing),
                     where=", ".join(sorted({f"{tt['name'].iloc[t_of_c[c - 1]]} t{tt['topo_number'].iloc[t_of_c[c - 1]]}"
                                             f" (set {tt['set'].iloc[t_of_c[c - 1]]})" for c in v.failing}))))
pd.DataFrame(rows).set_index("check")

,first_failing_step,failing,where
check,,,
no topounit sends water to itself,1,4,"caspian 184059 t4 (set 0), caspian 184059 t4 (set 1), lowest last t3 (set 0), lowest last t3 (set 1)"
D3 the outlet passes on at least what it does without IM2,1,6,"caspian 184059 t4 (set 0), caspian 184059 t4 (set 1), cycle t3 (set 0), cycle t3 (set 1), lowest last t3 (set 0), lowest last t3 (set 1)"


The failing elements are the outlets of the bug's hillslopes: the self-looping outlet of "lowest last"
and of the Caspian gridcell, and in D3 also "cycle"'s lowest topounit, which points uphill. "Lowest
first" and "flat" never fail. (While this spec was written, Hypothesis shrank a D3 failure to an
example with negative runoff at the outlet: a drawn water table at the surface (`zwt` = 0, the edge of
its rule) next to unrelated soil water makes ELM's `Drainage` return negative drainage in the first
step. That comes from the inconsistent start, not from IM2; the outlet still fails because with IM2 it
keeps none of its own surface runoff.)

## 5. Over a storm: how much reaches the river

As in the first notebook, "lowest last" gets the inputs of the element of "lowest first" at the same
elevation (same set), so the two are the same hillslope stored in two orders and only the topology
differs. Over 20 drawn storms, the cumulative runoff to the river after the day, relative to IM2 off:

In [11]:
elev = tt["elevation"].to_numpy()


def mirror(example, src="lowest first", dst="lowest last"):
    out = copy.deepcopy(example)
    ct = lay.table("c")
    maps = {}
    for level, table in (("t", tt), ("c", ct)):
        e = elev if level == "t" else elev[lay.ancestor_of("c", "t")]
        idx = np.arange(len(table))
        m = {}
        for k in sorted(set(table["set"])):
            a = idx[(table["name"] == src).to_numpy() & (table["set"] == k).to_numpy()]
            b = idx[(table["name"] == dst).to_numpy() & (table["set"] == k).to_numpy()]
            m.update({j: a[np.nonzero(e[a] == e[j])[0][0]] for j in b})
        maps[level] = m
    for name, arr in out.inputs.items():
        var = case.inv.vars.get(name)
        if var is None or var.level not in maps:
            continue
        x = np.moveaxis(np.asarray(arr), var.subgrid_axis, 0)
        for j, i in maps[var.level].items():
            x[j] = x[i]
        out.inputs[name] = np.moveaxis(x, 0, var.subgrid_axis)
    return out


twins = gt["name"].isin(["lowest first", "lowest last"]).to_numpy()
rows = []
for seed in range(20):
    ex2 = mirror(s.sample("storm", seed=seed))
    a, b = river(s.execute("storm", ex2)), river(off_session.execute("storm", ex2))
    half_on = (a >= 0.5 * a[-1]).argmax(axis=0) + 1   # step by which half of the day's runoff had arrived
    half_off = (b >= 0.5 * b[-1]).argmax(axis=0) + 1
    rows += [dict(seed=seed, set=gt["set"].iloc[i], hillslope=gt["name"].iloc[i], on=a[-1, i], off=b[-1, i],
                  half_on=half_on[i], half_off=half_off[i]) for i in np.nonzero(twins)[0]]
storms = pd.DataFrame(rows)
storms["on / off"] = storms["on"] / storms["off"]
print("IM2 off, the twins agree:", np.allclose(storms[storms.hillslope == "lowest first"]["off"].to_numpy(),
                                                storms[storms.hillslope == "lowest last"]["off"].to_numpy(),
                                                rtol=1e-12, atol=1e-15))
storms.groupby("hillslope")[["off", "on", "on / off", "half_off", "half_on"]].describe().loc[
    :, (slice(None), ["mean", "min", "max"])].T

IM2 off, the twins agree: True


hillslope      lowest first  lowest last
off      mean       59.8690   5.9869e+01
         min         0.5247   5.2474e-01
         max       267.6823   2.6768e+02
on       mean       58.5785   5.7803e+01
         min         0.1375   2.6363e-06
         max       266.1270   2.6549e+02
on / off mean        0.7761   6.2652e-01
         min         0.1746   3.1545e-06
         max         0.9980   9.9725e-01
half_off mean        8.6250   8.6250e+00
         min         1.0000   1.0000e+00
         max        40.0000   4.0000e+01
half_on  mean        9.7250   1.7450e+01
         min         1.0000   1.0000e+00
         max        41.0000   4.4000e+01

Twenty storms of mirrored twins (two sets each, 40 pairs). IM2 off, the twins agree to rounding, so
their difference is the topology alone. "half" is the step by which half of the day's runoff had
reached the river.

* With the intended neighbours ("lowest first"), on average 0.78 of the IM2-off runoff has reached the
  river by the end of the day (at least 0.17 in every storm), and half of it arrives by step 9.7
  on average against 8.6 without IM2: the stores delay the runoff by about half an hour and part of it
  infiltrates downhill.
* With the bug ("lowest last"), 0.63 on average, and in the worst storm 3e-6: essentially nothing.
  Half of it arrives only by step 17.5. What the outlet keeps recirculates through its own store into
  its own soil.

The totals include subsurface drainage, which IM2 does not route (and which the drawn soil states make
large here), so the ratios understate the effect on surface runoff.

## 6. What this shows

* Multi-step runs work on ELM's real hydrology: two routines, 48 steps, the state carried, the rest of
  the step stated in the spec (one `forcing`, one `between`, 59-60 `keep`), checked after every step.
* Every invariant of the IM2 store holds over a day whatever the topology: finite, non-negative,
  conserved, released at `frac_from_uphill` per step, and against IM2 off: identical where IM2 must not
  act (one-topounit gridcells, the topmost topounit's soil), total water conserved, never sooner to the
  river, never less water where it arrives.
* Only the checks that express what the routing is for fail, and only at the bug's outlets: no
  topounit sends to itself, and the outlet passes on at least what it does without IM2 (D3). D3 is the
  property ELM's code breaks: a self-looping outlet passes on less than with no routing at all.
* Over storms the bug withholds a further 15% of the day's runoff on average (and up to all of it) and
  nearly doubles the time half of it takes to reach the river (17.5 against 9.7 steps).

Not covered here: weight-0 downhill targets (H2: the store becomes Inf, then NaN) and topounits with no
hydrology column (H3: the NaN stores of the full ELM runs). Both need the `between` drain to ignore
columns the routines do not process (proptest sets their outputs to NaN), which is the next change.

In [12]:
s.close()